In [5]:
import os
import pickle
from time import time
import numpy as np
from collections import defaultdict
from tmu.models.autoencoder.autoencoder import TMAutoEncoder
from sklearn.metrics.pairwise import cosine_similarity
target_similarity=defaultdict(list)
from contextlib import redirect_stdout
from tqdm import tqdm
import re

def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    words = text.split()
    english_stopwords = set(stopwords.words('english'))
    words = [word for word in words if word not in english_stopwords]
    processed_text = ' '.join(words)
    return processed_text

clause_weight_threshold = 0
number_of_examples = 2000
accumulation = 25
factor = 4
clauses = 80
T = factor*40
s = 5.0
epochs = 25
involved_datasets = []
involved_datasets.append(["Combined",0,1])
target_words_clauses = []

f_vectorizer_X = open("vectorizer_X.pickle", "rb")
vectorizer_X = pickle.load(f_vectorizer_X)
f_vectorizer_X.close()
feature_names = vectorizer_X.get_feature_names_out()
number_of_features = vectorizer_X.get_feature_names_out().shape[0]

f_X = open("X.pickle", "rb")
X_train = pickle.load(f_X)
f_X.close()

folder_path = 'datasets'
dir_count = 0
for item in os.listdir(folder_path):
    item_path = os.path.join(folder_path, item)
    if os.path.isdir(item_path):
        dir_count += 1
progress_bar = tqdm(total=dir_count, desc="Running Datasets")

for folder_name in os.listdir(folder_path):
    if folder_name == 'wordsim353-sim':
        current_folder_path = os.path.join(folder_path, folder_name)
        if os.path.isdir(current_folder_path):
            files_start_name = os.path.join(current_folder_path, folder_name)
            with open(files_start_name + '_word1.pkl', 'rb') as f:
                word1 = pickle.load(f)
            with open(files_start_name + '_word2.pkl', 'rb') as f:
                word2 = pickle.load(f)
            word_total= list(set(word1 + word2))

            target_words=[]
            for i in word_total:
                if i in vectorizer_X.vocabulary_:
                    target_words.append(i)
            output= open(files_start_name + '_indivdual_target.pkl', "wb")
            pickle.dump(target_words, output)
            output.close()

            output_active = np.empty(len(target_words), dtype=np.uint32)
            for i in range(len(target_words)):
                target_word = target_words[i]
                target_id = vectorizer_X.vocabulary_[target_word]
                output_active[i] = target_id

            with open(files_start_name + '_indivdual_results.txt', 'w') as file, redirect_stdout(file):
                print("Loading dataset: " + folder_name)
                words_progress_bar = tqdm(total=len(output_active), desc="Running Words")
                output_active_list = output_active
                for i in output_active_list:
                    target_word_clauses = []
                    single_output_active = np.empty(1, dtype=np.uint32)
                    single_output_active[0] = i
                    tm = TMAutoEncoder(clauses, T, s, single_output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=True)
                        
                    print("Individual run for target word: %s" % vectorizer_X.get_feature_names_out()[i])
                    print("Epochs: %d" % epochs)
                    print("Example: %d" % number_of_examples)
                    print("Target words: %d" % len(target_words))
                    print("Accumulation: %d" % accumulation)
                    print("No of features: %d" % number_of_features)

                    total_training = 0
                    epochs_progress_bar = tqdm(total=epochs, desc="Running Epochs")
                    for e in range(epochs):
                        start_training = time()
                        tm.documents_fit(X_train, number_of_examples=number_of_examples, involved_datasets=involved_datasets)
                        stop_training = time()
                        total_training = total_training + (stop_training-start_training)
                        epochs_progress_bar.update(1)
                    epochs_progress_bar.close()

                    seconds = total_training
                    hours = seconds // 3600
                    minutes = (seconds % 3600) // 60
                    seconds = seconds % 60
                    print(f"Training duration: {hours} hours, {minutes} minutes, {seconds} seconds")

                    print("\n=====================================\nClauses\n=====================================")
                    for j in range(clauses):
                        print("Clause #%-2d " % (j), end=' ')
                        weight = 0
                        weight = tm.get_weight(0, j)
                        print("W:%-5d " % (weight), end=' ')
                        l = [] 
                        related_literals = []
                        number_of_literals = 0 
                        for k in range(tm.clause_bank.number_of_literals):
                            if tm.get_ta_action(j, k) == 1:
                                number_of_literals = number_of_literals + 1
                                related_literals.append(k)
                                if k < tm.clause_bank.number_of_features:
                                    l.append("%s(%d)" % (feature_names[k], tm.clause_bank.get_ta_state(j, k)))
                                else:
                                    l.append("¬%s(%d)" % (feature_names[k-tm.clause_bank.number_of_features], tm.clause_bank.get_ta_state(j, k)))
                        print(": No of features:%-6d" % (number_of_literals), end=" ==> ")
                        try:
                            print(" ^ ".join(l))
                        except UnicodeEncodeError:
                            print(" exception ")
                        target_word_clauses.append([weight, related_literals])
                        
                    target_words_clauses.append([i,target_word_clauses])
                    words_progress_bar.update(1)
                
                with open(files_start_name + '_phase1.txt', "wb") as phase1file:
                    pickle.dump((output_active, feature_names, number_of_features,target_words_clauses), phase1file)
                    
                words_progress_bar.close()
                progress_bar.update(1)
progress_bar.close()

C:\Users\ahmedkk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:347: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
